# Lab 4: Free Sentinel-2 imagery and Google Earth Engine

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

**What you will do**

1. Search the Sentinel-2 archive in Google Earth Engine (GEE) for clear scenes over each AOI.
2. Download small image chips for the tanks.
3. Test whether 10 m pixels can measure floating-roof fill levels, and compare them with PlanetScope (3 m) and SkySat (0.5 m).
4. Learn a sub-pixel method (profile fitting) that works much better than edge-finding at 10 m.
5. Run both methods on real chips.

**What you need**

- A Google Earth Engine account and a Cloud project (free for noncommercial use; register at earthengine.google.com).
- Labs 2 and 3 completed or at least read. This lab reuses their functions through `tank_utils.py`.
- Packages in `notebooks/requirements.txt`.

**Why this matters for policy:** Sentinel-2 is free, open and re-runnable by anyone. A result built on it can be checked by a journalist, a rival government or a student. A result built on commercial imagery often cannot. That difference changes how much weight an outside claim can carry in a policy debate.

## Concepts you need first

**Sentinel-2.** A pair of European Earth-observation satellites (run by ESA for the Copernicus program). Data is free. Sentinel-2A, 2B and 2C have flown; check ESA's pages for which satellites are active now. With two active satellites, a given spot is revisited every 2 to 5 days.

**Pixel sizes.** Blue (B2), green (B3), red (B4) and near-infrared (B8) are 10 m. Other bands are 20 m or 60 m. We use only the four 10 m bands.

**Overpass time.** Like PlanetScope, Sentinel-2 crosses around 10:30 local solar time. The sun is therefore as high as the table in Lab 3 shows. You cannot choose a better time.

**Surface reflectance (SR) vs. top-of-atmosphere (TOA).** SR has been corrected for haze and gases. We use SR ("Level-2A").

**Google Earth Engine.** A Google service that stores satellite archives and runs your analysis on Google's computers. You write Python; the heavy work happens remotely.

**Server-side vs. client-side.** An `ee.Image` or `ee.ImageCollection` is not data. It is a *recipe* that lives on Google's servers. Nothing is computed until you ask for a result, usually with `.getInfo()` (small results) or a download URL. Beginners often print an `ee` object and get a description, not numbers. That is normal.

**Cloud Score+.** A Google dataset that rates how clear each Sentinel-2 pixel is, from 0 (cloud or shadow) to 1 (clear). We use its `cs_cdf` band to decide if a scene is usable over our small AOI. This is better than the scene-wide cloud percentage, because the scene covers 100 km by 100 km and our AOI is 5 km by 5 km.

**Sun angles.** Each Sentinel-2 image carries `MEAN_SOLAR_ZENITH_ANGLE` (angle from straight up) and `MEAN_SOLAR_AZIMUTH_ANGLE`. Sun elevation = 90° minus the zenith angle.

**The core problem at 10 m.** One pixel of shadow width stands for `10 m × tan(sun elevation)` of roof depth. At 30° that is 5.8 m, which is a third or more of a typical tank's height. The edge method from Lab 3 cannot resolve that.

**The sub-pixel idea.** A shadow that covers part of a pixel still makes that pixel darker. The *amount* of darkness tells you how much of the pixel is shaded. Instead of looking for an edge, we fit a model of the tank's brightness profile (lit roof, shaded roof, shaded ground) and adjust the shadow lengths until the model matches the pixels. This is called **profile fitting**.

**What profile fitting needs.** It works best when you know the tank's wall height and the brightness ratio between lit and shaded surfaces. Those are assumptions you must state. You will test how wrong assumptions change the answer.

## Setup: Install the packages

Run the next cell once before anything else. This lab needs several packages that may not be in your Python environment yet. If one is missing, the import cell below stops with `ModuleNotFoundError`. (For example, `No module named 'skimage'` means scikit-image is not installed.)

Two things to know:

- **Use `%pip`, not `!pip`.** The `%pip` command installs into the same Python environment that this notebook's kernel is running. `!pip` can install into a different Python, and the import will still fail.
- **Some package names differ from their import names.** `scikit-image` is imported as `skimage`, and `earthengine-api` is imported as `ee`. The install cell uses the install names.

The packages are the same ones listed in `notebooks/requirements.txt`. Lab 3's `tank_utils.py` needs `scikit-image`, which Labs 3 and 4 use to find tanks in images. Step 2 of this lab needs `earthengine-api`.

If the install cell reports that it installed something new, **restart the kernel** (Kernel menu, or the restart button in your editor) and then run the cells from the top. Python does not see newly installed packages until the kernel restarts.

In [ ]:
# %pip installs into the environment this notebook's kernel is using (unlike !pip).
# -q keeps the output short. Packages that are already installed are skipped quickly.
%pip install -q \
    numpy pandas matplotlib requests \
    shapely geopandas pyproj rasterio \
    scipy \
    scikit-image \
    earthengine-api
# Notes on the list above:
#   shapely, geopandas, pyproj, rasterio -> geometry, tables with shapes, map projections, GeoTIFF files
#   scipy                                -> curve fitting (least_squares) and image sampling
#   scikit-image                         -> imported as "skimage"; used by tank_utils.py for tank detection
#   earthengine-api                      -> imported as "ee"; the Google Earth Engine client

In [1]:
import os
import json
import math
import time
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import shape
from scipy.optimize import least_squares
from scipy.ndimage import map_coordinates

import ee   # the Google Earth Engine Python library

# Functions from Lab 3, saved in tank_utils.py so we do not copy them again.
from tank_utils import (render_scene, to_pixel, measure_fill, detect_tanks,
                        depth_per_pixel, sun_elevation_at, BARRELS_PER_M3)

ModuleNotFoundError: No module named 'skimage'

## Step 1: Settings

In [ ]:
REPO = Path("..").resolve()
AOI_DIR = REPO / "AOI"
OUT_DIR = REPO / "data" / "imagery"
TANK_DIR = REPO / "data" / "tanks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TANK_DIR.mkdir(parents=True, exist_ok=True)

# Search window.
START_DATE = "2025-10-01"
END_DATE = "2026-10-01"

# Earth Engine dataset names.
S2_SR = "COPERNICUS/S2_SR_HARMONIZED"                    # Sentinel-2 surface reflectance
CLOUD_SCORE = "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED" # per-pixel clarity score
QA_BAND = "cs_cdf"
BANDS = ["B2", "B3", "B4", "B8"]                          # blue, green, red, near-infrared (10 m)

# Filters.
MAX_TILE_CLOUD = 40        # first pass: scene-wide cloud percent (coarse)
CLEAR_THRESHOLD = 0.60     # a pixel counts as clear if cs_cdf is at least this
MIN_AOI_CLEAR = 0.95       # share of the AOI that must be clear

# Rules for picking scenes (same logic as Lab 1).
MIN_SUN_ELEV = 15
MAX_SUN_ELEV = 45
SCENES_PER_SITE = 2

# Tank assumptions. State these in any result you report.
ASSUMED_HEIGHT_M = 15      # ASSUMPTION: wall height used by profile fitting
MIN_PIXELS_ACROSS = 8
PIXEL_S2_M = 10.0
MIN_DIAMETER_S2_M = MIN_PIXELS_ACROSS * PIXEL_S2_M    # 80 m

DOWNLOAD_CHIPS = True      # set False to search only

## Step 2: Connect to Earth Engine

The first time, `ee.Authenticate()` opens a browser window so you can sign in with your Google account. It saves a token on your computer. After that, `ee.Initialize()` is enough.

Your **project id** is the name of your Google Cloud project that is registered for Earth Engine. Never share your token file.

In [ ]:
# Read the project id from an environment variable, or ask for it.
EE_PROJECT = os.environ.get("EE_PROJECT") or input("Earth Engine project id: ")

try:
    ee.Initialize(project=EE_PROJECT)    # works if you have signed in before
except Exception:
    ee.Authenticate()                    # opens a browser the first time
    ee.Initialize(project=EE_PROJECT)

# Quick test: ask the server for one number. This is a client-to-server round trip.
print("Connected. Test value from the server:", ee.Number(1).add(1).getInfo())

## Step 3: Load the AOIs

In [ ]:
aois = {}
for path in sorted(AOI_DIR.glob("*_aoi.geojson")):
    feature = json.loads(path.read_text())["features"][0]
    slug = path.stem.replace("_aoi", "")
    aois[slug] = {
        "name": feature["properties"]["facility"],
        "lat": feature["properties"]["center_lat"],
        "lon": feature["properties"]["center_lon"],
        "shape": shape(feature["geometry"]),
        # An Earth Engine geometry built from the same coordinates.
        "ee_geom": ee.Geometry.Polygon(feature["geometry"]["coordinates"]),
    }
print(len(aois), "sites loaded")

## Step 4: Search the Sentinel-2 archive

The recipe below:

1. starts from the Sentinel-2 SR collection,
2. keeps images that overlap the AOI and fall in the date range,
3. drops scenes that are very cloudy overall (a quick first cut),
4. attaches the Cloud Score+ band to each image,
5. measures what share of *our AOI* is clear in each image,
6. pulls the metadata into a table.

Only step 6 transfers data to your computer.

In [ ]:
META_PROPS = ["system:time_start", "MEAN_SOLAR_AZIMUTH_ANGLE",
              "MEAN_SOLAR_ZENITH_ANGLE", "CLOUDY_PIXEL_PERCENTAGE", "SPACECRAFT_NAME"]


def fetch_catalog(aoi_geom):
    """Return a table of Sentinel-2 scenes over one AOI, with clear share and sun angles."""
    cloud_score = ee.ImageCollection(CLOUD_SCORE)

    collection = (ee.ImageCollection(S2_SR)
                  .filterBounds(aoi_geom)
                  .filterDate(START_DATE, END_DATE)
                  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_TILE_CLOUD))
                  .linkCollection(cloud_score, [QA_BAND]))     # adds the cs_cdf band

    def add_clear_share(img):
        # 1 where the pixel is clear, 0 where it is not; the mean is the clear share.
        clear = img.select(QA_BAND).gte(CLEAR_THRESHOLD)
        share = clear.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi_geom,
                                   scale=20, maxPixels=int(1e8)).get(QA_BAND)
        return img.set("aoi_clear_share", share)

    collection = collection.map(add_clear_share)

    # Turn each image's metadata into a Feature so we can fetch them all at once.
    # "system:index" is a reserved name: Earth Engine treats it as the feature's id and
    # leaves it out of the properties. So we copy it into an ordinary property, "scene_id".
    props = META_PROPS + ["aoi_clear_share"]
    features = collection.map(
        lambda im: ee.Feature(None, im.toDictionary(props)).set("scene_id", im.get("system:index")))
    rows = ee.FeatureCollection(features).getInfo()["features"]   # server -> client
    table = pd.DataFrame([r["properties"] for r in rows])
    if table.empty:
        return table

    table["acquired"] = pd.to_datetime(table["system:time_start"], unit="ms", utc=True)
    table["sun_elevation"] = 90 - table["MEAN_SOLAR_ZENITH_ANGLE"]   # zenith -> elevation
    table["sun_azimuth"] = table["MEAN_SOLAR_AZIMUTH_ANGLE"]
    table = table.rename(columns={"scene_id": "id", "CLOUDY_PIXEL_PERCENTAGE": "tile_cloud_pct",
                                  "SPACECRAFT_NAME": "satellite"})
    keep = ["id", "acquired", "satellite", "tile_cloud_pct", "aoi_clear_share",
            "sun_elevation", "sun_azimuth"]
    return table[keep].sort_values("acquired").reset_index(drop=True)

In [ ]:
catalogs = {}
for slug, aoi in aois.items():
    table = fetch_catalog(aoi["ee_geom"])
    catalogs[slug] = table
    table.to_csv(OUT_DIR / f"s2_catalog_{slug}.csv", index=False)
    n_clear = int((table["aoi_clear_share"] >= MIN_AOI_CLEAR).sum()) if len(table) else 0
    print(f"{aoi['name']:<35} {len(table):>4} scenes, {n_clear:>3} mostly clear over the AOI")

## Step 5: Look at the results

Compare this chart with the one in Lab 1. Sentinel-2 and PlanetScope cross at nearly the same local time, so the sun angles should look alike. The main differences are how many scenes there are (Sentinel-2 has far fewer) and the pixel size.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, (slug, table) in zip(axes.ravel(), catalogs.items()):
    if len(table):
        clear = table["aoi_clear_share"] >= MIN_AOI_CLEAR
        ax.scatter(table.loc[~clear, "acquired"], table.loc[~clear, "sun_elevation"],
                   s=10, color="lightgray", label="cloudy over AOI")
        ax.scatter(table.loc[clear, "acquired"], table.loc[clear, "sun_elevation"],
                   s=14, color="tab:blue", label="clear")
    ax.axhspan(MIN_SUN_ELEV, MAX_SUN_ELEV, color="green", alpha=0.1)
    ax.set_title(aois[slug]["name"], fontsize=10)
    ax.tick_params(axis="x", rotation=45)
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_ylabel("Sun elevation (degrees)")
axes[1, 0].set_ylabel("Sun elevation (degrees)")
plt.tight_layout()
plt.show()

## Step 6: Pick scenes and download chips

In [ ]:
def pick_best(table):
    """Clear scenes with the sun inside the target range, lowest sun first."""
    keep = table[(table["aoi_clear_share"] >= MIN_AOI_CLEAR)
                 & table["sun_elevation"].between(MIN_SUN_ELEV, MAX_SUN_ELEV)]
    return keep.sort_values("sun_elevation").head(SCENES_PER_SITE)


selected = {}
for slug, table in catalogs.items():
    best = pick_best(table) if len(table) else table
    selected[slug] = best["id"].tolist()
    print(f"\n{aois[slug]['name']}")
    print(best[["id", "acquired", "aoi_clear_share", "sun_elevation"]].to_string(index=False)
          if len(best) else "  no scenes pass the filters")

(OUT_DIR / "s2_selected_scenes.json").write_text(json.dumps(selected, indent=2))

**Downloading from Earth Engine.** `getDownloadURL` asks the server to build a GeoTIFF for a region and returns a web link. The request is limited to about 32 MB. A 5 km by 5 km chip with 4 bands at 10 m is about 2 MB, so it fits easily.

We ask for the image in the local UTM projection so pixels are square and in meters. That is what Lab 3's code expects.

In [ ]:
def utm_epsg(lon, lat):
    """EPSG code of the UTM zone that contains a point."""
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"


def download_chip(index, aoi_geom, epsg, dest_file):
    """Download one Sentinel-2 image, cut to the AOI, as a GeoTIFF."""
    image = ee.Image(f"{S2_SR}/{index}").select(BANDS)
    url = image.getDownloadURL({"region": aoi_geom, "scale": 10, "crs": epsg,
                                "format": "GEO_TIFF"})
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)


for slug, ids in selected.items():
    if not ids or not DOWNLOAD_CHIPS:
        continue
    aoi = aois[slug]
    folder = OUT_DIR / slug
    folder.mkdir(parents=True, exist_ok=True)
    epsg = utm_epsg(aoi["lon"], aoi["lat"])
    table = catalogs[slug].set_index("id")
    for index in ids:
        tif = folder / f"s2_{index}_clip.tif"
        if tif.exists():                           # skip files we already have
            continue
        download_chip(index, aoi["ee_geom"], epsg, tif)
        # Save the sun angles next to the image so Step 9 can find them.
        sidecar = {"id": index,
                   "sun_azimuth": float(table.loc[index, "sun_azimuth"]),
                   "sun_elevation": float(table.loc[index, "sun_elevation"])}
        tif.with_suffix(".json").write_text(json.dumps(sidecar))
        print("saved", tif.name)

## Step 7: What can 10 m pixels resolve?

First the geometry, with no simulation. The table shows how much roof depth one pixel of shadow width stands for, for each sensor, at each site in winter, spring/fall and summer. It also shows the smallest tank diameter each sensor can use (8 pixels across).

In [ ]:
rows = []
for slug, aoi in aois.items():
    for label, month in [("Dec", 12), ("Mar/Sep", 3), ("Jun", 6)]:
        el = float(sun_elevation_at(aoi["lat"], month))
        rows.append({"site": aoi["name"], "month": label, "sun_elev": round(el, 0),
                     "SkySat 0.5 m": round(float(depth_per_pixel(0.5, el)), 2),
                     "PlanetScope 3 m": round(float(depth_per_pixel(3.0, el)), 2),
                     "Sentinel-2 10 m": round(float(depth_per_pixel(10.0, el)), 2)})
depth_table = pd.DataFrame(rows)
print("Roof depth (m) represented by one pixel of shadow width")
print(depth_table.pivot(index="site", columns="month",
                        values=["SkySat 0.5 m", "PlanetScope 3 m", "Sentinel-2 10 m"]).to_string())
print()
for name, pix in [("SkySat", 0.5), ("PlanetScope", 3.0), ("Sentinel-2", 10.0)]:
    print(f"{name}: smallest tank diameter at {MIN_PIXELS_ACROSS} pixels across = {MIN_PIXELS_ACROSS * pix:.0f} m")

In [ ]:
# How many of the tanks found in Lab 2 are big enough for Sentinel-2?
for slug, aoi in aois.items():
    f = TANK_DIR / f"tanks_{slug}.geojson"
    if not f.exists():
        print(f"{aoi['name']:<35} no Lab 2 file")
        continue
    d = gpd.read_file(f)["diameter_m"]
    print(f"{aoi['name']:<35} {int((d >= MIN_DIAMETER_S2_M).sum()):>3} of {len(d):>3} tanks are {MIN_DIAMETER_S2_M:.0f} m or wider")

## Step 8: A better method for coarse pixels

Lab 3's `measure_fill` finds where each shadow *ends*. With 10 m pixels, a shadow edge falls inside a pixel, and that pixel is only partly dark. Edge-finding rounds the edge to the nearest pixel, which gives a large, one-sided error.

**Profile fitting** uses all the brightness values, including partly dark pixels. The steps:

1. Read the brightness along the line through the tank center in the shadow direction (the profile).
2. Estimate two reference brightness levels from the image: lit ground (the middle brightness of the area around the tank) and lit roof (a high percentile of pixels inside the tank).
3. Set the shaded levels as fixed fractions of the lit levels: `roof_ratio` and `ground_ratio`. These two numbers are **calibration constants**. The defaults come from our simulation. On real imagery you must tune them. Step 8b shows what happens if they are wrong.
4. Build a model profile: lit ground, then shaded roof (inner shadow, length `L_in`), then lit roof, then shaded ground (outer shadow, length `L_out`), then lit ground. Blur it the way a real sensor would.
5. Change `L_in` until the model matches the measured profile as closely as possible (least squares).
6. If the wall height is known, set `L_out = H / tan(e)` and fit only `L_in`. This is the most stable option. If the height is unknown, fit both.
7. Fill = 1 - L_in ÷ L_out.

In [ ]:
def fit_fill_profile(img, col, row, radius_px, sun_az, sun_elev, pixel_m, height_m=None,
                     roof_ratio=0.2, ground_ratio=0.26, blur_px=0.8):
    """Estimate fill fraction by fitting a model brightness profile (works at coarse pixels).

    height_m: wall height. If given, the outer shadow length is computed from it.
    roof_ratio, ground_ratio: shaded brightness as a share of lit brightness.
    blur_px: blur of the sensor in pixels (a mix of optics and resampling).
    """
    R = radius_px

    # --- Reference brightness levels, measured from the image itself ---------
    yy, xx = np.indices(img.shape)
    dist = np.hypot(xx - col, yy - row)                       # distance from tank center
    lit_ground = np.median(img[(dist > 2 * R) & (dist < 4 * R)])   # ground around the tank
    lit_roof = np.percentile(img[dist <= 0.8 * R], 90)             # brightest part of the roof
    shade_roof, shade_ground = lit_roof * roof_ratio, lit_ground * ground_ratio

    # --- Sample the measured profile along the shadow direction --------------
    az_s = np.radians((sun_az + 180) % 360)
    dcol, drow = np.sin(az_s), -np.cos(az_s)
    tan_e = np.tan(np.radians(sun_elev))
    L_out_max = (height_m / tan_e / pixel_m) if height_m else 5 * R    # pixels
    t = np.arange(-1.5 * R, R + 1.6 * L_out_max + 3, 0.25)            # pixels from center
    measured = map_coordinates(img, [row + t * drow, col + t * dcol], order=1, mode="nearest")

    # --- The model: a clean step profile, then blurred ------------------------
    t_fine = np.arange(t[0] - 4, t[-1] + 4, 0.05)
    kx = np.arange(-int(4 * blur_px / 0.05), int(4 * blur_px / 0.05) + 1) * 0.05
    kernel = np.exp(-0.5 * (kx / blur_px) ** 2)
    kernel /= kernel.sum()                                            # blur kernel adds to 1

    def model(L_in, L_out):
        ideal = np.where(t_fine < -R, lit_ground,                     # ground before the tank
                np.where(t_fine < -R + L_in, shade_roof,              # inner shadow
                np.where(t_fine < R, lit_roof,                        # lit roof
                np.where(t_fine < R + L_out, shade_ground,            # outer shadow
                         lit_ground))))                               # ground beyond
        return np.interp(t, t_fine, np.convolve(ideal, kernel, mode="same"))

    # --- Fit ------------------------------------------------------------------
    if height_m:
        L_out = L_out_max
        grid = np.linspace(0, 2 * R, 161)                             # coarse search first
        start = grid[int(np.argmin([np.sum((model(L, L_out) - measured) ** 2) for L in grid]))]
        fit = least_squares(lambda p: model(p[0], L_out) - measured, [start], bounds=([0], [2 * R]))
        L_in = fit.x[0]
    else:
        best = min(((np.sum((model(a, b) - measured) ** 2), a, b)
                    for a in np.linspace(0, 2 * R, 21) for b in np.linspace(1, L_out_max, 15)))
        fit = least_squares(lambda p: model(p[0], p[1]) - measured, [best[1], best[2]],
                            bounds=([0, 0.5], [2 * R, L_out_max]))
        L_in, L_out = fit.x

    flags = []
    if L_in > 1.95 * R:
        flags.append("roof_fully_shaded_fill_is_upper_bound")
    if lit_roof / lit_ground < 1.1:
        flags.append("low_roof_contrast")

    fill = float(np.clip(1 - L_in / L_out, 0, 1))
    H = height_m if height_m else L_out * pixel_m * tan_e
    volume_m3 = fill * np.pi * (R * pixel_m) ** 2 * H
    return {"fill": fill, "L_in_m": L_in * pixel_m, "L_out_m": L_out * pixel_m,
            "height_m": H, "volume_m3": volume_m3, "barrels": volume_m3 * BARRELS_PER_M3,
            "flags": ";".join(flags), "t": t, "profile": measured, "model": model(L_in, L_out)}

### Try it on one simulated tank

A 100 m wide, 15 m tall tank, 40 percent full, seen at 10 m pixels with the sun at 30°. It is only 10 pixels across.

In [ ]:
SUN_AZ, SUN_EL = 135, 30
TRUE_FILL = 0.40
tank = {"x": 0, "y": 0, "diameter_m": 100, "height_m": 15, "fill": TRUE_FILL}
HALF, PIX = 300, 10.0

img = render_scene([tank], SUN_AZ, SUN_EL, PIX, HALF, seed=5)
col, row = to_pixel(0, 0, PIX, HALF)
R_px = tank["diameter_m"] / 2 / PIX

edge = measure_fill(img, col, row, R_px, SUN_AZ, SUN_EL, PIX, height_m=15)    # Lab 3 method
fit = fit_fill_profile(img, col, row, R_px, SUN_AZ, SUN_EL, PIX, height_m=15)  # new method

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Sentinel-2-like image (10 m pixels)")
axes[0].axis("off")
axes[1].plot(fit["t"] * PIX, fit["profile"], "o-", markersize=3, label="measured profile")
axes[1].plot(fit["t"] * PIX, fit["model"], label="fitted model")
axes[1].axvline(-50, color="gray", linestyle=":")
axes[1].axvline(50, color="gray", linestyle=":")
axes[1].set_xlabel("Distance from tank center along shadow direction (m)")
axes[1].set_ylabel("Brightness")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"True fill: {TRUE_FILL:.2f}")
print(f"Edge method (Lab 3):   {edge['fill']:.2f}")
print(f"Profile fitting:       {fit['fill']:.2f}")

# One image can mislead. Repeat with the tank at five different positions
# inside a pixel (different noise each time) to see the spread.
print("\nSame tank, five random placements:")
for seed in range(5):
    r = np.random.default_rng(seed)
    ox, oy = r.uniform(-PIX / 2, PIX / 2, 2)
    tk = dict(tank, x=ox, y=oy)
    im = render_scene([tk], SUN_AZ, SUN_EL, PIX, HALF, seed=seed)
    c, rw = to_pixel(ox, oy, PIX, HALF)
    e = measure_fill(im, c, rw, R_px, SUN_AZ, SUN_EL, PIX, height_m=15)["fill"]
    f = fit_fill_profile(im, c, rw, R_px, SUN_AZ, SUN_EL, PIX, height_m=15)["fill"]
    print(f"  placement {seed}: edge {e:.2f} | fit {f:.2f} | true {TRUE_FILL:.2f}")

## Step 8a: How much better is it, by sensor?

Same test as Lab 3, Step 4. We use the same 100 m tank for all three sensors so the comparison is fair. For each case we compare three methods:

- **edge:** Lab 3's method.
- **fit, height known:** profile fitting with the correct wall height.
- **fit, height unknown:** profile fitting that also fits the outer shadow.

The simulation gives the fit method a perfect set of calibration constants. Step 8b removes that advantage.

In [ ]:
rng = np.random.default_rng(7)
sensors = {"SkySat 0.5 m": 0.5, "PlanetScope 3 m": 3.0, "Sentinel-2 10 m": 10.0}
sun_elevs = [20, 30, 45]
fills = [0.1, 0.3, 0.5, 0.7, 0.9]
REPS = 3
HALF_SWEEP = 300

records = []
for sensor, pix in sensors.items():
    for el in sun_elevs:
        for f in fills:
            for rep in range(REPS):
                ox, oy = rng.uniform(-pix / 2, pix / 2, 2)        # random shift inside a pixel
                tk = {"x": ox, "y": oy, "diameter_m": 100, "height_m": 15, "fill": f}
                im = render_scene([tk], SUN_AZ, el, pix, HALF_SWEEP, seed=int(rng.integers(1e6)))
                c, r = to_pixel(ox, oy, pix, HALF_SWEEP)
                rad = 50 / pix
                records.append({
                    "sensor": sensor, "sun_elev": el, "true": f,
                    "edge": measure_fill(im, c, r, rad, SUN_AZ, el, pix, height_m=15)["fill"],
                    "fit, height known": fit_fill_profile(im, c, r, rad, SUN_AZ, el, pix, height_m=15)["fill"],
                    "fit, height unknown": fit_fill_profile(im, c, r, rad, SUN_AZ, el, pix)["fill"],
                })
sweep = pd.DataFrame(records)
methods = ["edge", "fit, height known", "fit, height unknown"]
for m in methods:
    sweep["err_" + m] = sweep[m] - sweep["true"]
print(len(sweep), "test cases")

In [ ]:
rmse = pd.DataFrame({m: sweep.groupby(["sensor", "sun_elev"])["err_" + m]
                        .apply(lambda e: np.sqrt((e.dropna() ** 2).mean())) for m in methods})
bias = pd.DataFrame({m: sweep.groupby(["sensor", "sun_elev"])["err_" + m].mean() for m in methods})
print("RMSE of fill fraction (smaller is better)")
print(rmse.round(3).to_string())
print("\nMean error (positive = reads tanks too full)")
print(bias.round(3).to_string())

# Chart for the 30 degree case.
sub = rmse.xs(30, level="sun_elev").loc[list(sensors)]
sub.plot(kind="bar", figsize=(8, 4))
plt.ylabel("RMSE of fill fraction")
plt.title("Error by sensor and method (sun 30 degrees, 100 m tank)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Reading the results.** In the simulation at 10 m, the edge method has an RMSE of about 0.14 to 0.37 depending on sun angle, and it always reads tanks too full. Profile fitting with a known wall height cuts that to about 0.04 to 0.09. That is the difference between unusable and a rough estimate. The same method also improves PlanetScope results, which suggests Lab 3's edge method is not the best choice at 3 m either.

**Height unknown is a weak fallback.** When the fit also has to find the outer shadow, the results are erratic. They can be worse than the edge method, even at fine pixels. A coarse sensor needs outside knowledge (here, the wall height) to make up for missing detail.

Be careful with these numbers. The simulated tank is clean and isolated, and the calibration constants are exact. Real tanks have roof fittings, neighbors, rim vents and haze.

## Step 8b: What if the assumptions are wrong?

Two checks on the Sentinel-2 case (10 m pixels, sun at 30°, 100 m tank):

1. The assumed wall height is off by -30 to +30 percent.
2. The shaded brightness ratios are off by -40 to +40 percent.

In [ ]:
def run_cases(height_factor=1.0, ratio_factor=1.0, n=30, seed=11):
    """Return (RMSE, mean error) for fill when assumptions are scaled by the given factors."""
    rg = np.random.default_rng(seed)
    errs = []
    for _ in range(n):
        f = rg.uniform(0.1, 0.9)
        ox, oy = rg.uniform(-5, 5, 2)
        tk = {"x": ox, "y": oy, "diameter_m": 100, "height_m": 15, "fill": f}
        im = render_scene([tk], SUN_AZ, 30, 10.0, 300, seed=int(rg.integers(1e6)))
        c, r = to_pixel(ox, oy, 10.0, 300)
        res = fit_fill_profile(im, c, r, 5.0, SUN_AZ, 30, 10.0, height_m=15 * height_factor,
                               roof_ratio=0.2 * ratio_factor, ground_ratio=0.26 * ratio_factor)
        errs.append(res["fill"] - f)
    errs = np.array(errs)
    return float(np.sqrt((errs ** 2).mean())), float(errs.mean())


rows = []
for hf in [0.7, 0.85, 1.0, 1.15, 1.3]:
    rows.append({"change": "wall height", "factor": hf, **dict(zip(["rmse", "bias"], run_cases(height_factor=hf)))})
for rf in [0.6, 0.8, 1.0, 1.2, 1.4]:
    rows.append({"change": "shade ratios", "factor": rf, **dict(zip(["rmse", "bias"], run_cases(ratio_factor=rf)))})
pd.DataFrame(rows).round(3)

A wrong height shifts every tank in the same direction, so one well-known reference tank can correct it. Wrong shade ratios do similar damage. This is why published tank-volume estimates should state their calibration source.

## Step 9: Run it on real Sentinel-2 chips

This section uses the chips from Step 6. If none exist yet, it prints a message and skips.

In [ ]:
import rasterio
from pyproj import Transformer

SITE = "sk_energy_ulsan_south_korea"      # change to any AOI name without "_aoi"
BRIGHTNESS_BANDS = (1, 2, 3)               # B2, B3, B4 in the order we downloaded them

folder = OUT_DIR / SITE
chips = sorted(folder.glob("s2_*_clip.tif")) if folder.exists() else []
HAVE_CHIP = len(chips) > 0
print(f"{len(chips)} Sentinel-2 chips found for {SITE}")

In [ ]:
if HAVE_CHIP:
    tif_path = chips[0]                                       # change the index to try another
    sidecar = json.loads(tif_path.with_suffix(".json").read_text())
    sun_az, sun_el = sidecar["sun_azimuth"], sidecar["sun_elevation"]
    print("Scene:", sidecar["id"], "| sun azimuth", round(sun_az, 1), "| elevation", round(sun_el, 1))

    with rasterio.open(tif_path) as src:
        bands = src.read(BRIGHTNESS_BANDS).astype("float32")
        s2_img = bands.mean(axis=0)                           # average visible bands
        lo, hi = np.percentile(s2_img[s2_img > 0], [2, 98])   # stretch to 0 to 1
        s2_img = np.clip((s2_img - lo) / (hi - lo), 0, 1)
        transform, crs = src.transform, src.crs
        pix_s2 = abs(transform.a)
    print("Chip shape:", s2_img.shape, "| pixel size:", round(pix_s2, 1), "m")

    plt.figure(figsize=(7, 7))
    plt.imshow(s2_img, cmap="gray")
    plt.title(f"{SITE}: Sentinel-2 {sidecar['id']}")
    plt.axis("off")
    plt.show()
else:
    print("No chips yet. Run Steps 4 to 6 first.")

In [ ]:
if HAVE_CHIP:
    tank_file = TANK_DIR / f"tanks_{SITE}.geojson"
    to_image_crs = Transformer.from_crs("EPSG:4326", crs, always_xy=True)

    if tank_file.exists():
        inv = gpd.read_file(tank_file)
        inv = inv[inv["diameter_m"] >= MIN_DIAMETER_S2_M]       # only tanks wide enough
        xs, ys = to_image_crs.transform(inv["center_lon"].values, inv["center_lat"].values)
        cols, rows_ = ~transform * (xs, ys)
        targets = pd.DataFrame({"col": cols - 0.5, "row": rows_ - 0.5,
                                "radius_px": inv["diameter_m"].values / 2 / pix_s2, "source": "osm",
                                "osm_id": inv["osm_id"].values if "osm_id" in inv.columns else np.nan})
    else:
        print("No Lab 2 file for this site. Using image-based detection instead.")
        targets = detect_tanks(s2_img, pix_s2, min_d_m=MIN_DIAMETER_S2_M, max_d_m=160)
        targets["source"] = "detected"
    print(len(targets), "tanks to measure")

    results = []
    for i, tk in targets.iterrows():
        e = measure_fill(s2_img, tk["col"], tk["row"], tk["radius_px"], sun_az, sun_el, pix_s2,
                         height_m=ASSUMED_HEIGHT_M)
        f = fit_fill_profile(s2_img, tk["col"], tk["row"], tk["radius_px"], sun_az, sun_el, pix_s2,
                             height_m=ASSUMED_HEIGHT_M)
        results.append({"tank": i, "osm_id": tk.get("osm_id"), "source": tk["source"],
                        "diameter_m": round(2 * tk["radius_px"] * pix_s2, 0),
                        "fill_edge": e.get("fill"), "fill_fit": f["fill"],
                        "barrels_fit": f["barrels"], "flags": f["flags"]})
    s2_metrics = pd.DataFrame(results)
    s2_metrics.insert(0, "scene", sidecar["id"])
    out_csv = TANK_DIR / f"metrics_s2_{SITE}_{sidecar['id']}.csv"
    s2_metrics.to_csv(out_csv, index=False)
    print("Saved", out_csv)
    s2_metrics.round(2)

## Step 10: What about radar?

Optical imagery needs daylight and clear sky. Radar does not. Earth Engine also hosts Sentinel-1 radar (`COPERNICUS/S1_GRD`), free at 10 m. Published work on floating-roof tanks uses higher-resolution radar and a different signal (the way the radar bounces off the tank wall and roof edge), not shadows. That method is beyond this lab. See the project's feasibility report for the sources.

## Check your work

- Does your Step 9 table show the edge and fit methods disagreeing? In which direction? Does that match the simulation in Step 8a?
- Pick one tank. Look at its chip. Can you see the inner shadow by eye?

## Exercises

1. In Step 8a, change the tank diameter to 80 m, then 150 m. How does the Sentinel-2 error change? What is the smallest tank you would trust?
2. Run Step 7 for December and June. Which sites could support a Sentinel-2 measurement in June? Explain with the table.
3. In Step 8b, which assumption hurts more, a 30 percent wall-height error or a 40 percent shade-ratio error? What does that tell you about where to spend effort in the field?
4. Sentinel-2 revisits every few days and PlanetScope every day. In a 30-day window at one site, count the clear scenes from each (use Lab 1 and Step 4). How does this change what you can say about *changes* in inventory?
5. Write a short memo (200 words) for a policymaker: should an open, free method with a larger error carry weight against a commercial method with a smaller error? Consider cost, repeatability, licensing and who can check the work.